In [1]:
# ===== NOTEBOOK 1 - STEP 1 (fast version): look at the 3 datasets =====
import os, time
t0 = time.time()

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count(os.sep) - 2
    name = os.path.basename(root)
    # Big image folders: we only count, we don't open them (too many files)
    if name in ("jpeg", "all-mias", "all-mias.tar") or depth > 4:
        print("    " * depth + f"{name}/  ({len(dirs)} folders, {len(files)} files)  <- not opened")
        dirs[:] = []
        continue
    print("    " * depth + f"{name}/  ({len(files)} files)")
    for f in files[:5]:
        print("    " * (depth + 1) + f)

print(f"\nDone in {time.time() - t0:.0f} seconds")


input/  (0 files)
    datasets/  (0 files)
        awsaf49/  (0 files)
            cbis-ddsm-breast-cancer-image-dataset/  (0 files)
                csv/  (6 files)
                    calc_case_description_test_set.csv
                    calc_case_description_train_set.csv
                    dicom_info.csv
                    mass_case_description_test_set.csv
                    mass_case_description_train_set.csv
                jpeg/  (6774 folders, 0 files)  <- not opened
        kmader/  (0 files)
            mias-mammography/  (3 files)
                Info.txt
                all-mias.tar.gz
                all_mias_scans.h5
                all-mias/  (0 folders, 325 files)  <- not opened
        martholi/  (0 files)
            inbreast/  (1 files)
                inbreast.tgz

Done in 2 seconds


In [2]:
# ===== NOTEBOOK 1 - STEP 2: setup + ONE cleaning function for the 3 datasets =====
!pip install -q pydicom xlrd
import os, glob, tarfile
import numpy as np, pandas as pd, cv2, pydicom
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ---- Paths (found in Step 1) ----
IN       = "/kaggle/input/datasets"
MIAS_DIR = f"{IN}/kmader/mias-mammography"
CBIS_DIR = f"{IN}/awsaf49/cbis-ddsm-breast-cancer-image-dataset"
INB_TGZ  = f"{IN}/martholi/inbreast/inbreast.tgz"
OUT_DIR  = "/kaggle/working/merged"
IMG_SIZE = 448
os.makedirs(f"{OUT_DIR}/images", exist_ok=True)

# ---- Unzip INbreast (1-3 minutes) ----
INB_DIR = "/tmp/inbreast"
if not os.path.exists(INB_DIR):
    print("Unzipping INbreast...")
    with tarfile.open(INB_TGZ) as tar:
        tar.extractall(INB_DIR)
INB_XLS = glob.glob(f"{INB_DIR}/**/INbreast.xls", recursive=True)[0]
INB_DCM = glob.glob(f"{INB_DIR}/**/ALL-IMGS", recursive=True)[0]
print("INbreast ready ✅")

# ---- 1) Read any image (DICOM / PGM / JPG) -> grey image with values 0..1 ----
def read_image(path):
    if path.endswith(".dcm"):
        ds = pydicom.dcmread(path)
        img = ds.pixel_array.astype(np.float32)
        if ds.PhotometricInterpretation == "MONOCHROME1":   # inverted grey levels
            img = img.max() - img
    else:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    return (img - img.min()) / (img.max() - img.min() + 1e-8)

# ---- 2) Find the breast = the BIGGEST bright shape (removes labels, markers, noise) ----
def breast_mask(img):
    img8 = cv2.medianBlur((img * 255).astype(np.uint8), 5)
    _, binary = cv2.threshold(img8, 20, 255, cv2.THRESH_BINARY)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((15, 15), np.uint8))
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(img8)
    if contours:
        biggest = max(contours, key=cv2.contourArea)
        cv2.drawContours(mask, [biggest], -1, 255, thickness=cv2.FILLED)
    return mask > 0

# ---- 3) Full cleaning: mask -> orientation -> crop -> contrast -> resize ----
def clean_image(path, size=IMG_SIZE):
    img = read_image(path)
    h, w = img.shape
    scale = 1024 / max(h, w)                       # work at max 1024 px (faster)
    if scale < 1:
        img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

    mask = breast_mask(img)
    if mask.sum() < 0.05 * mask.size:              # mask failed -> keep whole image
        mask = np.ones_like(mask)
    img = img * mask                               # remove everything outside the breast

    half = mask.shape[1] // 2                      # orientation: breast on the LEFT
    if mask[:, :half].sum() < mask[:, half:].sum():
        img, mask = img[:, ::-1], mask[:, ::-1]

    ys, xs = np.where(mask)                        # crop to the breast
    img  = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    mask = mask[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    lo, hi = np.percentile(img[mask], [1, 99.5])   # contrast (breast pixels only)
    img = np.clip((img - lo) / (hi - lo + 1e-8), 0, 1) * mask

    img = cv2.resize(np.ascontiguousarray(img), (size, size), interpolation=cv2.INTER_AREA)
    return (img * 255).astype(np.uint8)

# ---- 4) TEST on 3 images of each dataset: RAW vs CLEAN ----
cbis_info = pd.read_csv(f"{CBIS_DIR}/csv/dicom_info.csv")
cbis_full = (cbis_info[cbis_info["SeriesDescription"] == "full mammogram images"]["image_path"]
             .str.replace("CBIS-DDSM/", f"{CBIS_DIR}/", regex=False))
samples = {
    "MIAS":      [f"{MIAS_DIR}/all-mias/mdb{n:03d}.pgm" for n in (1, 2, 28)],
    "INbreast":  sorted(glob.glob(f"{INB_DCM}/*.dcm"))[:3],
    "CBIS-DDSM": cbis_full.iloc[:3].tolist(),
}
fig, axes = plt.subplots(3, 6, figsize=(20, 11))
for r, (name, paths) in enumerate(samples.items()):
    for k, p in enumerate(paths):
        axes[r, 2 * k].imshow(read_image(p), cmap="gray")
        axes[r, 2 * k].set_title(f"{name} RAW", fontsize=10)
        axes[r, 2 * k + 1].imshow(clean_image(p), cmap="gray", vmin=0, vmax=255)
        axes[r, 2 * k + 1].set_title(f"{name} CLEAN", fontsize=10)
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Cleaning test: RAW vs CLEAN (labels removed, breast on the left, cropped, contrast)")
plt.tight_layout(); plt.show()


Unzipping INbreast...


/tmp/ipykernel_16/2993514407.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(INB_DIR)


INbreast ready ✅


[figure removed: it showed mammogram images, which cannot be redistributed]


In [3]:
# ===== NOTEBOOK 1 - STEP 3: labels of the 3 datasets in ONE table =====
LABELS  = {0: "Normal", 1: "Benign", 2: "Malignant"}
NAME2ID = {v: k for k, v in LABELS.items()}

# ---------------- MIAS (Info.txt) ----------------
rows = []
with open(f"{MIAS_DIR}/Info.txt", errors="ignore") as f:
    for line in f:
        t = line.split()
        if not t or not t[0].startswith("mdb"):
            continue
        sev = t[3] if len(t) > 3 and t[3] in ("B", "M") else "N"   # N = normal
        rows.append({"ref": t[0], "abn_class": t[2], "severity": sev})
mias_rows = pd.DataFrame(rows)
print(f"MIAS: {len(mias_rows)} rows for {mias_rows['ref'].nunique()} images (some images have 2 lesions)")

def mias_label(sev):
    if (sev == "M").any(): return "Malignant"
    if (sev == "B").any(): return "Benign"
    return "Normal"

mias = (mias_rows.groupby("ref")
        .agg(label_name=("severity", mias_label),
             original=("abn_class", lambda s: "+".join(sorted(set(s)))))
        .reset_index())
mias["dataset"]    = "MIAS"
mias["image_id"]   = mias["ref"]
mias["patient_id"] = "MIAS_" + ((mias["ref"].str[3:].astype(int) + 1) // 2).astype(str).str.zfill(3)
mias["path"]       = MIAS_DIR + "/all-mias/" + mias["ref"] + ".pgm"
mias["laterality"] = ""          # not given in MIAS
mias["view"]       = "MLO"       # all MIAS images are MLO views

# ---------------- INbreast (INbreast.xls) ----------------
xls = pd.read_excel(INB_XLS)
xls["image_id"] = pd.to_numeric(xls["File Name"], errors="coerce")
xls = xls.dropna(subset=["image_id"])
xls["image_id"] = xls["image_id"].astype(int).astype(str)
xls["birads"] = (xls["Bi-Rads"].astype(str).str.strip().str.lower()
                 .str.replace(r"\.0$", "", regex=True))
rows = []
for p in sorted(glob.glob(f"{INB_DCM}/*.dcm")):
    parts = os.path.basename(p).split("_")
    rows.append({"image_id": parts[0], "patient_id": "INB_" + parts[1], "path": p,
                 "laterality": parts[3], "view": "MLO" if parts[4].startswith("ML") else parts[4]})
inb = pd.DataFrame(rows).merge(xls[["image_id", "birads"]], on="image_id", how="left")
b = inb["birads"].str[0].astype(int)
inb["label_name"] = np.where(b <= 2, "Normal", np.where(b == 3, "Benign", "Malignant"))
inb["original"]   = "BI-RADS " + inb["birads"]
inb["dataset"]    = "INbreast"

# ---------------- CBIS-DDSM (mass cases, as in the course reference notebook) ----------------
mass = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f"{CBIS_DIR}/csv/mass_case_description_*.csv"))])
print(f"CBIS mass: {len(mass)} rows (1 row = 1 lesion) {mass['pathology'].value_counts().to_dict()}")
mass = mass[mass["pathology"] != "BENIGN_WITHOUT_CALLBACK"]          # as in the course reference notebook
uid2path = {p.split("/")[-2]: p for p in cbis_full}                  # folder name -> jpg path
mass["path"] = mass["image file path"].str.strip().str.split("/").str[2].map(uid2path)
print("CBIS rows without a jpg file:", int(mass["path"].isna().sum()))
mass = mass.dropna(subset=["path"])
cbis = (mass.groupby("path")                                          # 1 image = 1 row
        .agg(patient_id=("patient_id", "first"),
             laterality=("left or right breast", "first"),
             view=("image view", "first"),
             label_name=("pathology", lambda s: "Malignant" if (s == "MALIGNANT").any() else "Benign"),
             original=("pathology", lambda s: "+".join(sorted(set(s)))))
        .reset_index())
cbis["patient_id"] = "CBIS_" + cbis["patient_id"]
cbis["laterality"] = cbis["laterality"].str[0]
cbis["image_id"]   = cbis["path"].str.split("/").str[-2].str[-12:]
cbis["dataset"]    = "CBIS-DDSM"

# ---------------- ONE table for everything ----------------
cols = ["dataset", "image_id", "patient_id", "path", "laterality", "view", "label_name", "original"]
meta = pd.concat([mias[cols], inb[cols], cbis[cols]], ignore_index=True)
meta["label_multi"] = meta["label_name"].map(NAME2ID)                        # 0 Normal, 1 Benign, 2 Malignant
meta["label_bin"]   = meta["label_name"].map({"Benign": 0, "Malignant": 1}).fillna(-1).astype(int)  # -1 = Normal
meta["exists"]      = meta["path"].map(os.path.exists)

print(f"\nTotal: {len(meta)} images | {meta['patient_id'].nunique()} patients | missing files: {(~meta['exists']).sum()}")
print("\nIMAGES per dataset and class:")
display(pd.crosstab(meta["dataset"], meta["label_name"], margins=True))
print("PATIENTS per dataset and class:")
display(meta.pivot_table(index="dataset", columns="label_name", values="patient_id",
                         aggfunc="nunique", fill_value=0, margins=True))

meta.to_csv(f"{OUT_DIR}/metadata_raw.csv", index=False)


MIAS: 330 rows for 322 images (some images have 2 lesions)
CBIS mass: 1696 rows (1 row = 1 lesion) {'MALIGNANT': 784, 'BENIGN': 771, 'BENIGN_WITHOUT_CALLBACK': 141}
CBIS rows without a jpg file: 0

Total: 2205 images | 1091 patients | missing files: 0

IMAGES per dataset and class:


label_name,Benign,Malignant,Normal,All
dataset,,,,
CBIS-DDSM,726,747,0,1473
INbreast,23,100,287,410
MIAS,63,52,207,322
All,812,899,494,2205


PATIENTS per dataset and class:


label_name,Benign,Malignant,Normal,All
dataset,,,,
CBIS-DDSM,403,431,0,822
INbreast,10,49,96,108
MIAS,62,52,154,161
All,475,532,250,1091


In [4]:
# ===== NOTEBOOK 1 - STEP 4: clean ALL images (keep the real shape) and save as PNG =====
from joblib import Parallel, delayed

def clean_image_v2(path, size=IMG_SIZE):
    img = read_image(path)
    h, w = img.shape
    scale = 1024 / max(h, w)                          # work at max 1024 px (faster)
    if scale < 1:
        img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

    mask = breast_mask(img)                           # keep only the breast
    breast_frac = float(mask.mean())
    if breast_frac < 0.05:                            # mask failed -> keep whole image
        mask = np.ones_like(mask)
    img = img * mask

    half = mask.shape[1] // 2                         # breast on the LEFT
    if mask[:, :half].sum() < mask[:, half:].sum():
        img, mask = img[:, ::-1], mask[:, ::-1]

    ys, xs = np.where(mask)                           # crop
    img  = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    mask = mask[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    lo, hi = np.percentile(img[mask], [1, 99.5])      # contrast (breast pixels only)
    img = np.clip((img - lo) / (hi - lo + 1e-8), 0, 1) * mask

    # NEW: black padding to a square (breast stays on the left) -> keeps the REAL shape
    ch, cw = img.shape
    s = max(ch, cw)
    square = np.zeros((s, s), dtype=np.float32)
    top = (s - ch) // 2
    square[top:top + ch, :cw] = img

    out = cv2.resize(square, (size, size), interpolation=cv2.INTER_AREA)
    return (out * 255).astype(np.uint8), {"breast_frac": round(breast_frac, 3),
                                          "crop_h": ch, "crop_w": cw}

def process(i, row):
    try:
        img, info = clean_image_v2(row["path"])
        fname = f"images/{i:05d}_{row['dataset']}_{row['image_id']}.png"
        cv2.imwrite(f"{OUT_DIR}/{fname}", img)
        return {"file": fname, "ok": True, "error": "", **info}
    except Exception as e:
        return {"file": None, "ok": False, "error": str(e)[:120]}

# Run on 4 CPU cores in parallel
results = Parallel(n_jobs=4)(delayed(process)(i, row)
                             for i, row in tqdm(meta.iterrows(), total=len(meta), desc="Cleaning"))
meta = pd.concat([meta.reset_index(drop=True), pd.DataFrame(results)], axis=1)

print(f"Saved: {meta['ok'].sum()} images | errors: {(~meta['ok']).sum()}")
if (~meta["ok"]).any():
    display(meta.loc[~meta["ok"], ["dataset", "path", "error"]])

meta.to_csv(f"{OUT_DIR}/metadata.csv", index=False)

# Quick look: 8 random saved images
fig, axes = plt.subplots(1, 8, figsize=(22, 3.5))
for ax, (_, r) in zip(axes, meta[meta["ok"]].sample(8, random_state=1).iterrows()):
    ax.imshow(cv2.imread(f"{OUT_DIR}/{r['file']}", cv2.IMREAD_GRAYSCALE), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{r['dataset']}\n{r['label_name']}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()


Cleaning:   0%|          | 0/2205 [00:00<?, ?it/s]

Saved: 2205 images | errors: 0


[figure removed: it showed mammogram images, which cannot be redistributed]


In [5]:
# ===== NOTEBOOK 1 - STEP 5: QUALITY CHECK of all 2205 cleaned images =====
import hashlib

imgs = np.stack([cv2.imread(f"{OUT_DIR}/{f}", cv2.IMREAD_GRAYSCALE)
                 for f in tqdm(meta["file"], desc="Loading")])
flat   = imgs.reshape(len(imgs), -1)
inside = flat > 0                                            # breast pixels

meta["breast_area"] = inside.mean(1).round(3)                # % of the square covered by breast
meta["mean_px"] = [f[m].mean() if m.any() else 0 for f, m in zip(flat, inside)]   # brightness
meta["std_px"]  = [f[m].std()  if m.any() else 0 for f, m in zip(flat, inside)]   # contrast
meta["aspect"]  = (meta["crop_h"] / meta["crop_w"]).round(2)                     # height / width

# ---- 1) Duplicates: exact copies, and near-copies (16x16 thumbnail fingerprint) ----
meta["md5"]   = [hashlib.md5(x.tobytes()).hexdigest() for x in imgs]
small         = np.stack([cv2.resize(x, (16, 16), interpolation=cv2.INTER_AREA) for x in imgs])
meta["ahash"] = [np.packbits(s > s.mean()).tobytes().hex() for s in small]
exact = meta[meta.duplicated("md5", keep=False)]
near  = meta[meta.duplicated("ahash", keep=False)]
print(f"1) Exact duplicate images: {len(exact)} | near-duplicates: {len(near)}")
if len(near):
    display(near.sort_values("ahash")[["dataset", "patient_id", "image_id", "label_name", "ahash"]].head(20))

# ---- 2) Orientation: breast on the left for every image? ----
half = imgs.shape[2] // 2
left_ok = (imgs[:, :, :half] > 0).sum((1, 2)) >= (imgs[:, :, half:] > 0).sum((1, 2))
print(f"2) Breast on the left: {left_ok.sum()}/{len(imgs)}")

# ---- 3) Statistics per dataset ----
print("3) Statistics per dataset (mean):")
display(meta.groupby("dataset")[["breast_frac", "breast_area", "aspect", "mean_px", "std_px"]]
        .agg(["mean", "min", "max"]).round(2))

# ---- 4) Suspicious images (mask failed, breast too small/too big, strange shape) ----
sus = meta[(meta["breast_frac"] < 0.05) | (meta["breast_area"] < 0.12) |
           (meta["breast_area"] > 0.90) | (meta["aspect"] > 3) | (meta["aspect"] < 0.7)]
print(f"4) Suspicious images: {len(sus)}")
display(sus["dataset"].value_counts())
if len(sus):
    show = sus.head(16)
    fig, axes = plt.subplots(2, 8, figsize=(22, 6))
    for ax in axes.ravel(): ax.axis("off")
    for ax, (i, r) in zip(axes.ravel(), show.iterrows()):
        ax.imshow(imgs[i], cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"#{i} {r['dataset']}\narea={r['breast_area']} asp={r['aspect']}", fontsize=8)
    plt.suptitle("4) Suspicious images (to check by eye)"); plt.tight_layout(); plt.show()

# ---- 5) Shortcut check: do the 3 datasets look different? ----
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for d in ["MIAS", "INbreast", "CBIS-DDSM"]:
    m = meta["dataset"] == d
    ax[0].hist(meta.loc[m, "mean_px"], bins=40, alpha=0.5, density=True, label=d)
    ax[1].hist(meta.loc[m, "std_px"],  bins=40, alpha=0.5, density=True, label=d)
ax[0].set_title("Brightness of the breast (mean pixel)"); ax[0].legend()
ax[1].set_title("Contrast of the breast (std of pixels)"); ax[1].legend()
plt.suptitle("5) Do the 3 datasets look different? (big difference = shortcut risk)")
plt.tight_layout(); plt.show()
print("Mean brightness per dataset and class:")
display(meta.pivot_table(index="dataset", columns="label_name", values="mean_px", aggfunc="mean").round(1))

# ---- 6) 3 random images for each dataset x class ----
combos = [(d, c) for d in ["MIAS", "INbreast", "CBIS-DDSM"] for c in ["Normal", "Benign", "Malignant"]
          if ((meta["dataset"] == d) & (meta["label_name"] == c)).any()]
fig, axes = plt.subplots(len(combos), 3, figsize=(9, 3 * len(combos)))
for row, (d, c) in enumerate(combos):
    idx = meta[(meta["dataset"] == d) & (meta["label_name"] == c)].sample(3, random_state=0).index
    for k, i in enumerate(idx):
        axes[row, k].imshow(imgs[i], cmap="gray", vmin=0, vmax=255); axes[row, k].axis("off")
        axes[row, k].set_title(f"{d} | {c}", fontsize=9)
plt.suptitle("6) Random images per dataset and class"); plt.tight_layout(); plt.show()

meta.to_csv(f"{OUT_DIR}/metadata.csv", index=False)


Loading:   0%|          | 0/2205 [00:00<?, ?it/s]

1) Exact duplicate images: 0 | near-duplicates: 53


,dataset,patient_id,image_id,label_name,ahash
1562,CBIS-DDSM,CBIS_P_00454,920677800792,Benign,000080008000e000f000f000f800f800f800f800f000f0...
902,CBIS-DDSM,CBIS_P_00309,531105331307,Benign,000080008000e000f000f000f800f800f800f800f000f0...
1227,CBIS-DDSM,CBIS_P_00401,960607706763,Benign,0000e000f800fc00fe00ff00ff00ff80ff80ff80ff80ff...
375,INbreast,INB_301f1776aebbf5d2,22579870,Malignant,0000e000f800fc00fe00ff00ff00ff80ff80ff80ff80ff...
1688,CBIS-DDSM,CBIS_P_00653,321843261053,Malignant,800080008000c000e000f000f800f800f800f800f000f0...
1900,CBIS-DDSM,CBIS_P_01504,282525029721,Malignant,800080008000c000e000f000f800f800f800f800f000f0...
2142,CBIS-DDSM,CBIS_P_01671,300880140008,Malignant,80008000c000e000f000f000f800f800f800f800f800f0...
1076,CBIS-DDSM,CBIS_P_01886,213634425726,Benign,80008000c000e000f000f000f800f800f800f800f800f0...
624,INbreast,INB_1e4b534393d18753,50998349,Normal,8000c000e000f000f000f800f800f800f800f800f800f0...
513,INbreast,INB_f8bfddc28e8045c0,26933772,Normal,8000c000e000f000f000f800f800f800f800f800f800f0...


2) Breast on the left: 2205/2205
3) Statistics per dataset (mean):


breast_frac             breast_area             aspect              \
                 mean   min   max        mean   min   max   mean   min   max   
dataset                                                                        
CBIS-DDSM        0.44  0.07  1.00        0.28  0.08  0.66   2.17  1.16  5.36   
INbreast         0.33  0.09  0.58        0.35  0.14  0.57   2.17  1.32  3.78   
MIAS             0.38  0.14  0.90        0.41  0.16  0.89   2.07  1.02  4.13   

          mean_px                 std_px                
             mean     min     max   mean    min    max  
dataset                                                 
CBIS-DDSM  120.72   51.16  203.30  53.56  25.92  81.12  
INbreast   131.46   47.31  179.12  47.12  29.36  65.20  
MIAS       151.53  109.48  196.03  63.12  37.75  83.53

4) Suspicious images: 177


dataset
CBIS-DDSM    144
MIAS          22
INbreast      11
Name: count, dtype: int64

Mean brightness per dataset and class:


label_name,Benign,Malignant,Normal
dataset,,,
CBIS-DDSM,118.6,122.8,NaN
INbreast,138.9,127.1,132.4
MIAS,151.4,153.2,151.2


[figure removed: it showed mammogram images, which cannot be redistributed]


In [6]:
# ===== NOTEBOOK 1 - STEP 6: FIX the 2 problems, clean again, check again =====

# ---- Fix 1: remove film borders / white lines stuck to the breast ----
def breast_mask_v3(img):
    img8 = cv2.medianBlur((img * 255).astype(np.uint8), 5)
    m = max(3, int(0.015 * max(img8.shape)))           # NEW: erase a thin frame at the image border
    img8[:m, :] = 0; img8[-m:, :] = 0; img8[:, :m] = 0; img8[:, -m:] = 0
    _, binary = cv2.threshold(img8, 20, 255, cv2.THRESH_BINARY)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((25, 25), np.uint8))  # stronger: cuts thin lines
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(img8)
    if contours:
        cv2.drawContours(mask, [max(contours, key=cv2.contourArea)], -1, 255, thickness=cv2.FILLED)
    return mask > 0

# ---- Fix 2: histogram equalization INSIDE the breast (same brightness for all datasets) ----
def equalize_breast(img, mask):
    q = np.clip((img * 1023).astype(np.int32), 0, 1023)        # 1024 grey levels
    hist = np.bincount(q[mask], minlength=1024).astype(np.float64)
    cdf = hist.cumsum() / max(hist.sum(), 1)
    return np.where(mask, 0.05 + 0.95 * cdf[q], 0).astype(np.float32)   # breast 0.05..1, background 0

def clean_image_v3(path, size=IMG_SIZE):
    img = read_image(path)
    h, w = img.shape
    scale = 1024 / max(h, w)
    if scale < 1:
        img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

    mask = breast_mask_v3(img)
    breast_frac = float(mask.mean())
    if breast_frac < 0.05:
        mask = np.ones_like(mask)

    half = mask.shape[1] // 2                                  # breast on the LEFT
    if mask[:, :half].sum() < mask[:, half:].sum():
        img, mask = img[:, ::-1], mask[:, ::-1]

    ys, xs = np.where(mask)                                    # crop
    img  = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    mask = mask[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    img = equalize_breast(img, mask)                           # NEW: same brightness distribution

    ch, cw = img.shape                                         # black padding -> real shape
    s = max(ch, cw)
    square = np.zeros((s, s), dtype=np.float32)
    top = (s - ch) // 2
    square[top:top + ch, :cw] = img
    out = cv2.resize(square, (size, size), interpolation=cv2.INTER_AREA)
    return (out * 255).astype(np.uint8), {"breast_frac": round(breast_frac, 3), "crop_h": ch, "crop_w": cw}

def process_v3(i, row):
    try:
        img, info = clean_image_v3(row["path"])
        cv2.imwrite(f"{OUT_DIR}/{row['file']}", img)           # same file name -> replaces the old one
        return {"ok": True, "error": "", **info}
    except Exception as e:
        return {"ok": False, "error": str(e)[:120]}

# ---- Clean all images again (about 15 minutes) ----
res = Parallel(n_jobs=4)(delayed(process_v3)(i, row)
                         for i, row in tqdm(meta.iterrows(), total=len(meta), desc="Cleaning v3"))
res = pd.DataFrame(res)
for c in ["ok", "error", "breast_frac", "crop_h", "crop_w"]:
    meta[c] = res[c].values
print(f"Saved: {meta['ok'].sum()} | errors: {(~meta['ok']).sum()}")

# ---- New quality report ----
imgs   = np.stack([cv2.imread(f"{OUT_DIR}/{f}", cv2.IMREAD_GRAYSCALE) for f in tqdm(meta["file"], desc="Loading")])
flat   = imgs.reshape(len(imgs), -1)
inside = flat > 0
meta["breast_area"] = inside.mean(1).round(3)
meta["mean_px"] = [f[m].mean() if m.any() else 0 for f, m in zip(flat, inside)]
meta["std_px"]  = [f[m].std()  if m.any() else 0 for f, m in zip(flat, inside)]
meta["aspect"]  = (meta["crop_h"] / meta["crop_w"]).round(2)

# Better rule: the mask took (almost) the whole image, or found almost nothing
sus = meta[(meta["breast_frac"] > 0.80) | (meta["breast_frac"] < 0.07)].sort_values("breast_frac", ascending=False)
print(f"Suspicious images (mask failed): {len(sus)}", sus["dataset"].value_counts().to_dict())
if len(sus):
    fig, axes = plt.subplots(2, 8, figsize=(22, 6))
    for ax in axes.ravel(): ax.axis("off")
    for ax, (i, r) in zip(axes.ravel(), sus.head(16).iterrows()):
        ax.imshow(imgs[i], cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"#{i} {r['dataset']}\nfrac={r['breast_frac']}", fontsize=8)
    plt.suptitle("Suspicious images after the fix"); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for d in ["MIAS", "INbreast", "CBIS-DDSM"]:
    m = meta["dataset"] == d
    ax[0].hist(meta.loc[m, "mean_px"], bins=40, alpha=0.5, density=True, label=d)
    ax[1].hist(meta.loc[m, "std_px"],  bins=40, alpha=0.5, density=True, label=d)
ax[0].set_title("Brightness AFTER equalization"); ax[0].legend()
ax[1].set_title("Contrast AFTER equalization"); ax[1].legend()
plt.tight_layout(); plt.show()

# Same random images as before -> compare with Step 5
combos = [(d, c) for d in ["MIAS", "INbreast", "CBIS-DDSM"] for c in ["Normal", "Benign", "Malignant"]
          if ((meta["dataset"] == d) & (meta["label_name"] == c)).any()]
fig, axes = plt.subplots(len(combos), 3, figsize=(9, 3 * len(combos)))
for row, (d, c) in enumerate(combos):
    idx = meta[(meta["dataset"] == d) & (meta["label_name"] == c)].sample(3, random_state=0).index
    for k, i in enumerate(idx):
        axes[row, k].imshow(imgs[i], cmap="gray", vmin=0, vmax=255); axes[row, k].axis("off")
        axes[row, k].set_title(f"{d} | {c}", fontsize=9)
plt.suptitle("Random images AFTER the fixes (same images as Step 5)"); plt.tight_layout(); plt.show()

meta.to_csv(f"{OUT_DIR}/metadata.csv", index=False)


Cleaning v3:   0%|          | 0/2205 [00:00<?, ?it/s]

Saved: 2205 | errors: 0


Loading:   0%|          | 0/2205 [00:00<?, ?it/s]

Suspicious images (mask failed): 3 {'MIAS': 2, 'CBIS-DDSM': 1}


[figure removed: it showed mammogram images, which cannot be redistributed]


In [7]:
# ===== NOTEBOOK 1 - STEP 7: final dataset + summary =====
REMOVE = [1856]   # CBIS image: mask found only 5% of the image (breast cut) -> removed after visual check

for i in REMOVE:
    f = f"{OUT_DIR}/{meta.loc[i, 'file']}"
    if os.path.exists(f):
        os.remove(f)
meta["keep"] = meta["ok"] & ~meta.index.isin(REMOVE)
final = meta[meta["keep"]].copy()

cols = ["file", "dataset", "patient_id", "image_id", "laterality", "view",
        "label_name", "label_multi", "label_bin", "original", "breast_frac", "crop_h", "crop_w"]
final[cols].to_csv(f"{OUT_DIR}/metadata_final.csv", index=False)

print(f"FINAL DATASET: {len(final)} images | {final['patient_id'].nunique()} patients\n")

print("MULTICLASS task (Normal / Benign / Malignant): images")
display(pd.crosstab(final["dataset"], final["label_name"], margins=True))

binary = final[final["label_bin"] >= 0]
print(f"BINARY task (Benign / Malignant): {len(binary)} images | {binary['patient_id'].nunique()} patients")
display(pd.crosstab(binary["dataset"], binary["label_name"], margins=True))

# README: how the data was made (reproducibility)
readme = f"""Merged mammography dataset (MIAS + INbreast + CBIS-DDSM) - TP01 CNN
Images: {len(final)} PNG, {IMG_SIZE}x{IMG_SIZE}, grey (1 channel)
Labels: Normal (MIAS N, INbreast BI-RADS 1-2) | Benign (MIAS B, INbreast BI-RADS 3, CBIS BENIGN)
        | Malignant (MIAS M, INbreast BI-RADS 4-6, CBIS MALIGNANT)
CBIS: mass cases only, BENIGN_WITHOUT_CALLBACK removed, 1 image = 1 row (Malignant if any lesion is malignant)
MIAS: 1 image = 1 row (Malignant if any lesion is malignant), patient = pair (mdb001+mdb002)
Cleaning: breast mask (biggest shape, border frame erased) -> breast on the left -> crop
          -> histogram equalization inside the breast -> black padding to square -> resize
Removed after visual check: {len(REMOVE)} image(s)
Columns: label_multi (0 Normal, 1 Benign, 2 Malignant), label_bin (0 Benign, 1 Malignant, -1 Normal = not used)
Split must be done BY PATIENT (column patient_id).
"""
with open(f"{OUT_DIR}/README.txt", "w") as f:
    f.write(readme)

size_mb = sum(os.path.getsize(os.path.join(r, x)) for r, _, fs in os.walk(OUT_DIR) for x in fs) / 1e6
print(f"Output folder: {OUT_DIR}  ({size_mb:.0f} MB)")
print(os.listdir(OUT_DIR))


FINAL DATASET: 2204 images | 1091 patients

MULTICLASS task (Normal / Benign / Malignant): images


label_name,Benign,Malignant,Normal,All
dataset,,,,
CBIS-DDSM,726,746,0,1472
INbreast,23,100,287,410
MIAS,63,52,207,322
All,812,898,494,2204


BINARY task (Benign / Malignant): 1710 images | 988 patients


label_name,Benign,Malignant,All
dataset,,,
CBIS-DDSM,726,746,1472
INbreast,23,100,123
MIAS,63,52,115
All,812,898,1710


Output folder: /kaggle/working/merged  (105 MB)
['metadata.csv', 'metadata_raw.csv', 'images', 'metadata_final.csv', 'README.txt']


In [8]:
# ===== NOTEBOOK 1 - STEP 8: pack everything into FEW files =====
# Kaggle has problems with thousands of small files (limit of 500 files when creating a dataset),
# so all images go into ONE compressed file. Row i of the file = row i of metadata_final.csv
import shutil
final = pd.read_csv(f"{OUT_DIR}/metadata_final.csv")
X_all = np.stack([cv2.imread(f"{OUT_DIR}/{f}", cv2.IMREAD_GRAYSCALE)
                  for f in tqdm(final["file"], desc="Packing")])

PACK_DIR = "/kaggle/working/packed"
os.makedirs(PACK_DIR, exist_ok=True)
np.savez_compressed(f"{PACK_DIR}/images_{IMG_SIZE}.npz", X=X_all)
final.to_csv(f"{PACK_DIR}/metadata_final.csv", index=False)
shutil.copy(f"{OUT_DIR}/README.txt", PACK_DIR)

shutil.rmtree(OUT_DIR)          # remove the 2204 small PNG files from the output
for f in sorted(os.listdir(PACK_DIR)):
    print(f"{f:25s} {os.path.getsize(f'{PACK_DIR}/{f}') / 1e6:8.1f} MB")
print("Images:", X_all.shape, "✅")


Packing:   0%|          | 0/2204 [00:00<?, ?it/s]

README.txt                     0.0 MB
images_448.npz               146.0 MB
metadata_final.csv             0.2 MB
Images: (2204, 448, 448) ✅
